# 📰 Fake News Detection using Deep Learning

This project tackles the problem of misinformation by automatically classifying news articles as **REAL or FAKE**
using Natural Language Processing (NLP) and Deep Learning techniques.

The project is structured as a progressive pipeline starting from classical machine learning with TF-IDF features,
and gradually moving towards more powerful transformer-based models. Each model builds on the previous one,
making it easy to compare how much improvement deep learning brings over traditional approaches.

---

## 📂 Dataset
Two datasets were used `True.csv` containing real news articles and `Fake.csv` containing fake ones.
After labeling, deduplication, and cleaning, both were merged into a single DataFrame of **33,700 articles**.
The cleaned dataset was saved as `cleaned_text_with_token_reduction.csv` for reuse across all models.

---

## 🧹 Preprocessing
Raw text was cleaned using a SpaCy-based pipeline that removed URLs, digits, and punctuation,
followed by lemmatization and lowercasing. The final data was split 80/20 into training and test sets
with stratification to maintain class balance.

---

## 🤖 Models Built

**Model 1 — Logistic Regression (TF-IDF, C=1)**
A simple baseline using TF-IDF vectorization (5000 features, unigrams + bigrams).
Achieved **99% accuracy** and ROC-AUC of **0.9989** — surprisingly strong for a classical model.

**Model 2 — Logistic Regression (TF-IDF, Tuned C=10)**
Cross-validation across C = [0.01, 0.1, 1, 10] found C=10 as optimal.
Improved ROC-AUC to **0.9994** with tighter, more consistent performance.

**Model 3 — DistilBERT Embeddings + Logistic Regression**
CLS embeddings (768-dim) were extracted from a pre-trained DistilBERT model and passed into Logistic Regression.
Achieved **99.3% test accuracy** — contextual embeddings gave a slight edge over raw TF-IDF.

**Model 4 — Fine-Tuned DistilBERT (End-to-End)**
`DistilBertForSequenceClassification` was fine-tuned directly on the dataset for 4 epochs with AdamW optimizer,
linear warmup scheduler, and class-weighted loss. Only **4 misclassifications** out of 6,732 samples —
effectively **~100% accuracy**.

---

## 📊 Results Summary

| Model | Test Accuracy | ROC-AUC |
|---|---|---|
| LR + TF-IDF (C=1) | 0.99 | 0.9989 |
| LR + TF-IDF (Tuned C=10) | 0.9918 | 0.9994 |
| DistilBERT Embeddings + LR | 0.9930 | — |
| **Fine-Tuned DistilBERT** | **~1.00** | — |

---

## 🛠️ Tech Stack
Python · Pandas · NumPy · SpaCy · NLTK · Scikit-learn · HuggingFace Transformers · PyTorch · Google Colab

---

**Author:** Muhammad Saad Umar — BS Information Technology, Bahauddin Zakariya University, Multan
**GitHub:** [Fake-News-Detection-using-Deep-Learning](https://github.com/Saadumar26/Fake-News-Detection-using-Deep-Learning)

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [ ]:
# === 1. Import Libraries ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
import string

# Download NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
folder_path = '/content/drive/My Drive/FYP/'


## Loading **True** & **Fake** CSV's

In [ ]:
# === 2. Load Data ===
true_df = pd.read_csv('/content/drive/My Drive/FYP/True.csv')
fake_df = pd.read_csv('/content/drive/My Drive/FYP/Fake.csv')

true_df['label'] = 1
fake_df['label'] = 0

## Pre-process True Dataset

In [ ]:
true_df.head()

In [ ]:
true_df.isnull().sum()

In [ ]:
true_df.shape

In [ ]:
true_df.duplicated().sum()

In [ ]:
true_df.drop_duplicates(inplace=True)
true_df.duplicated().sum()

In [ ]:
true_df.shape

## Final validation code for **true_df**

In [ ]:
# 1. Check for missing values
print("Missing values per column:\n", true_df.isnull().sum())

In [ ]:
# 2. Check for fully duplicated rows (should be 0)
print("\nFully duplicated rows:", true_df.duplicated().sum())

In [ ]:
# 3. Check for duplicated news text only
if 'text' in true_df.columns:
    print("Duplicated 'text' entries:", true_df.duplicated(subset='text').sum())
else:
    print("'text' column not found! Please confirm the column name.")

In [ ]:
# 4. Check unique labels (should be only 1)
print("\nUnique labels in 'label' column:", true_df['label'].unique())

In [ ]:
# 5. Check for any label not equal to 1
print("Any rows with label not 1?:", (true_df['label'] != 1).sum())

In [ ]:
# 6. Reset index for cleanliness
true_df = true_df.reset_index(drop=True)
print("\nFinal shape of clean `true_df`:", true_df.shape)

In [ ]:
# See duplicated text entries (first few)
dup_texts = true_df[true_df.duplicated(subset='text', keep=False)]
print(dup_texts)

In [ ]:
# Print the number of duplicated entries
print(f"Total duplicated rows based on 'text': {len(dup_texts)}")

In [ ]:
# Remove all rows where 'text' is duplicated (including the first occurrence)
true_df_cleaned = true_df[~true_df.duplicated(subset='text', keep=False)].copy()

# Optional: Reset index
true_df_cleaned.reset_index(drop=True, inplace=True)

# Check result
print(f"Original rows: {len(true_df)}")
print(f"Rows after removing duplicates: {len(true_df_cleaned)}")


In [ ]:
dup_texts_with_title = true_df[true_df.duplicated(subset=['title', 'text'])]
print(dup_texts_with_title)

In [ ]:
# Print the number of duplicated entries
print(f"Total duplicated rows based on 'text': {len(dup_texts_with_title)}")

In [ ]:
true_df_cleaned

## Pre-process Fake Dataset

In [ ]:
fake_df.head()

In [ ]:
fake_df.isnull().sum()

In [ ]:
fake_df.shape

In [ ]:
fake_df = fake_df[fake_df['text'].notnull() & (fake_df['text'].str.strip() != '')]

In [ ]:
print("Original shape:", fake_df.shape)
cleaned_fake_df = fake_df[fake_df['text'].notnull() & (fake_df['text'].str.strip() != '')]
print("After cleaning:", cleaned_fake_df.shape)


In [ ]:
# Find rows where text length after strip is 0
mask = fake_df['text'].str.strip().str.len() == 0
print("Rows with empty/whitespace text:", mask.sum())


In [ ]:
fake_df = fake_df[fake_df['text'].notnull() & (fake_df['text'].str.strip().str.len() > 0)]


In [ ]:
print("Original shape:", fake_df.shape)
cleaned_fake_df = fake_df[fake_df['text'].notnull() & (fake_df['text'].str.strip().str.len() > 0)]
print("After cleaning:", cleaned_fake_df.shape)

In [ ]:
cleaned_fake_df.duplicated().sum()

In [ ]:
cleaned_fake_df.drop_duplicates(inplace=True)
cleaned_fake_df.duplicated().sum()

In [ ]:
cleaned_fake_df.shape

In [ ]:
print(cleaned_fake_df['text'].isnull().sum())  # should be 0


In [ ]:
print((cleaned_fake_df['text'].str.strip().str.len() == 0).sum())  # should be 0


In [ ]:
cleaned_fake_df['text'].sample(5)  # random 5 samples to confirm non-empty text


In [ ]:
# Final fake_df cleaning
cleaned_fake_df = fake_df[fake_df['text'].notnull() & (fake_df['text'].str.strip().str.len() > 0)].copy()
cleaned_fake_df = cleaned_fake_df[~cleaned_fake_df.duplicated(subset='text', keep=False)].copy()
cleaned_fake_df.reset_index(drop=True, inplace=True)

print(f"Final cleaned fake_df shape: {cleaned_fake_df.shape}")

In [ ]:
cleaned_fake_df

In [ ]:
# Save cleaned CSV
cleaned_fake_df.to_csv('/content/drive/My Drive/FYP/Fake_cleaned.csv', index=False)

# DataFrame
Merge **true_df**, **cleaned_fake_df** into the DataFrame

In [ ]:
df = pd.concat([true_df_cleaned, cleaned_fake_df]).reset_index(drop=True)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

## Double Check for Duplicate Samples

In [ ]:
# Check for exact duplicate rows (text + label)
duplicate_rows = df.duplicated(subset=['text', 'label']).sum()
print(f"Number of duplicate text+label rows: {duplicate_rows}")

# Check for duplicate texts only (regardless of label)
duplicate_texts = df.duplicated(subset='text').sum()
print(f"Number of duplicate texts (ignoring label): {duplicate_texts}")


In [ ]:
df = df.drop_duplicates(subset='text').reset_index(drop=True)
df

## Visualize the Subject of News

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
df.groupby('subject').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

## Visualize the Label

In [ ]:
from matplotlib import pyplot as plt
df['label'].plot(kind='hist', bins=20, title='label')
plt.gca().spines[['top', 'right',]].set_visible(False)

## Visualize the data distribution

In [ ]:
print(df['label'].unique())


In [ ]:
df['label'] = df['label'].map({0: 'FAKE', 1: 'TRUE'})


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
ax = sns.countplot(data=df, x='label', palette='Set2')

# Add count labels on top of each bar
for p in ax.patches:
    count = int(p.get_height())
    ax.text(p.get_x() + p.get_width() / 2., count + 5,   # position
            str(count), ha="center", fontsize=12)

plt.title('Distribution of True vs. Fake News')
plt.xlabel('News Type')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


## Data Cleaning Pipeline
In the journey to detect fake news, **clarity begins with clean data.** This preprocessing pipeline was crafted to strip away the noise — URLs, digits, punctuation, and stopwords — while preserving the **semantic core** of the text. By leveraging **SpaCy's linguistic intelligence**, it isolates meaningful words like **nouns, verbs, adjectives, and proper nouns**, and transforms them into their base forms. This process isn't just cleaning — it's **refining raw language into structured insight**, laying a strong foundation for accurate, context-aware model predictions. Each cleaned token now holds the **essence of meaning**, empowering the model to learn not just from words, but from **relevant and purposeful expression.**

In [ ]:
import time
import re
import string
import spacy
from tqdm import tqdm

start_time = time.time()

# Load SpaCy English model (disable unnecessary components for speed)
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

# 1️⃣ Basic text cleaning before SpaCy
def basic_clean(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)  # remove URLs
    text = re.sub(r'\d+', '', text)                     # remove digits
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply basic cleaning first
df['pre_cleaned_text'] = df['text'].apply(basic_clean)

# 2️⃣ SpaCy processing function (lemmatize + lowercase)
def clean_text_spacy(doc):
    tokens = [
        token.lemma_.lower()
        for token in doc
        if token.is_alpha
    ]
    return ' '.join(tokens)

# Enable tqdm progress bar for pandas (optional)
tqdm.pandas()

# 3️⃣ Batch process with SpaCy pipe (with progress bar)
docs = list(tqdm(
    nlp.pipe(df['pre_cleaned_text'], batch_size=500, n_process=1),
    total=len(df),
    desc="Processing Text"
))

# 4️⃣ Apply cleaning function
df['clean_text'] = [clean_text_spacy(doc) for doc in docs]

# Drop intermediate column
df.drop(columns=['pre_cleaned_text'], inplace=True)

end_time = time.time()
minutes = int((end_time - start_time) // 60)
seconds = int((end_time - start_time) % 60)

print(f"Time taken for cleaning text (batch + lemmatization): {minutes} minutes and {seconds} seconds")

In [ ]:
df['clean_text']

In [ ]:
df['clean_text'][0]

In [ ]:
print(df.shape)
print(df.columns)


## Comparing **Raw** vs. **Cleaned Text**
This transformation sharpens the data, empowering models to detect truth with clarity.
---



In [ ]:
df[['text', 'clean_text']].head()


In [ ]:
# Show original and cleaned text for the first row (index 0)
original_text = df.loc[0, 'text']
cleaned_text = df.loc[0, 'clean_text']

# Print both for clear comparison
print("=== ORIGINAL TEXT ===")
print(original_text)
print("\n=== CLEANED TEXT ===")
print(cleaned_text)


In [ ]:
folder_path = '/content/drive/My Drive/FYP/'

df.to_csv(folder_path + 'cleaned_text_with_token_reduction.csv', index=False)

print("The cleaned DataFrame has been saved to 'cleaned_text_with_token_reduction.csv'")


## Token Reduction

In [ ]:
import matplotlib.pyplot as plt

# Tokenize text before cleaning
df['tokens_before_cleaning'] = df['text'].apply(lambda x: len(x.split()))

# Tokenize text after cleaning (after lemmatization, stopword removal, etc.)
df['tokens_after_cleaning'] = df['clean_text'].apply(lambda x: len(x.split()))

# Calculate the reduction in tokens
df['token_reduction'] = df['tokens_before_cleaning'] - df['tokens_after_cleaning']

# Plot the comparison
plt.figure(figsize=(10, 6))
plt.bar(df.index[:100], df['token_reduction'][:100], color='skyblue')
plt.title('Token Reduction (Before vs After Cleaning) - Sample of First 100 Rows')
plt.xlabel('Sample Index')
plt.ylabel('Tokens Reduced')
plt.show()

# Display token count statistics
total_reduction = df['token_reduction'].sum()
print(f"Total Tokens Reduced: {total_reduction}")
print(f"Average Tokens Reduced per Row: {df['token_reduction'].mean():.2f}")


## Splitting the Data

In [ ]:
# Should be done BEFORE TF-IDF
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, stratify=df['label'], random_state=42
)

## Vectorization
In the quest to separate real from fake, raw text alone isn't enough — we need a way to **quantify meaning.** TF-IDF (Term Frequency–Inverse Document Frequency) steps in as a powerful bridge, converting cleaned words into **numerical vectors** that reflect their true importance across the dataset. By focusing on the most relevant unigrams and bigrams, it ensures **that frequent yet distinctive terms** carry more weight, while common, less informative ones fade away. This transformation equips the model with **structured, meaningful features,** enabling it to learn not just from words, but from their **contextual significance —** paving the way for smarter, sharper predictions in the fight against fake news.

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), max_df=0.95, min_df=2)
X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

In [ ]:
# Check the shape of the resulting matrix (rows = samples, columns = features)
print(f"Shape of the training feature matrix (X_train): {X_train.shape}")
print(f"Shape of the testing feature matrix (X_test): {X_test.shape}")
print(f"Shape of the training labels (y_train): {y_train.shape}")
print(f"Shape of the testing labels (y_test): {y_test.shape}")


In [ ]:
# Check a few feature names to see what terms are being used
feature_names = tfidf.get_feature_names_out()
print(f"Some of the feature names: {feature_names[:20]}")  # Display the first 20 features

# 🔹 Model 1: Logistic Regression (TF-IDF Features)
With Balanced Class

In [ ]:
from sklearn.linear_model import LogisticRegression

# Initialize and train the Logistic Regression model with class balancing
model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model.fit(X_train, y_train)

## Model Coefficient
After training the machine learning model for Fake News Detection, it is helpful to inspect the shape of the learned coefficients using the **model.coef_** attribute. These coefficients represent the weights that the model assigns to each textual feature (such as TF-IDF tokens) to distinguish between real and fake news. By printing the shape of this matrix, we can confirm how many features the model has learned from the dataset. Since this is a binary classification task—classifying news as either "Fake" or "Real"—the shape is typically (1, number_of_features), indicating one set of weights corresponding to the single decision boundary. Displaying this information verifies that the model has been successfully trained and offers insight into how it interprets different textual patterns to detect fake news.

In [ ]:
print(f"Model training completed. Coefficients shape: {model.coef_.shape}")

In [ ]:
# Predict on the test set
y_pred = model.predict(X_test)

print(f"Prediction completed. Number of predictions made: {len(y_pred)}")

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Classification report
print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix:\n{cm}")

# Plotting confusion matrix as heatmap
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Fake', 'True'], yticklabels=['Fake', 'True'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Predict labels
y_pred = model.predict(X_test)

# If original labels were strings
y_test_labels = y_test if isinstance(y_test.iloc[0], str) else y_test.map({1: 'TRUE', 0: 'FAKE'})
y_pred_labels = pd.Series(y_pred).map({1: 'TRUE', 0: 'FAKE'}) if isinstance(y_pred[0], int) else y_pred

# Classification report
print("=== Classification Report ===")
print(classification_report(y_test_labels, y_pred_labels))

# Confusion matrix
print("=== Confusion Matrix ===")
print(confusion_matrix(y_test_labels, y_pred_labels))


## Feature Importance Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Get feature names and model coefficients
feature_names = tfidf.get_feature_names_out()
coefs = model.coef_[0]  # 1D array since binary classification

# Get top 15 positive (TRUE) and top 15 negative (FAKE) coefficients
top_n = 15
top_positive_indices = np.argsort(coefs)[-top_n:]   # Most positive weights
top_negative_indices = np.argsort(coefs)[:top_n]    # Most negative weights

# Combine and label
top_indices = np.hstack([top_negative_indices, top_positive_indices])
top_features = [feature_names[i] for i in top_indices]
top_weights = coefs[top_indices]
colors = ['red' if w < 0 else 'green' for w in top_weights]

# Sort for better visualization
sorted_indices = np.argsort(top_weights)
top_features = [top_features[i] for i in sorted_indices]
top_weights = top_weights[sorted_indices]
colors = [colors[i] for i in sorted_indices]

# Plot
plt.figure(figsize=(10, 8))
bars = plt.barh(top_features, top_weights, color=colors)
plt.title("Top 30 TF-IDF Features Influencing Fake vs. True News (Logistic Regression)")
plt.xlabel("Logistic Regression Coefficient")
plt.ylabel("TF-IDF Feature")

# Add labels on bars
for i, v in enumerate(top_weights):
    plt.text(v, i, f"{v:.2f}", va='center', ha='right' if v < 0 else 'left', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# Map labels: 'TRUE' → 1, 'FAKE' → 0
y_test_numeric = y_test.map({'TRUE': 1, 'FAKE': 0}).astype(int)

# Get predicted probabilities for class 1 (TRUE news)
y_probs = model.predict_proba(X_test)[:, 1]

# Compute ROC AUC score
roc_auc = roc_auc_score(y_test_numeric, y_probs)
print(f"ROC-AUC Score: {roc_auc:.4f}")

# Compute ROC curve
fpr, tpr, thresholds = roc_curve(y_test_numeric, y_probs)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, linewidth=2, label=f'Logistic Regression (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')  # Diagonal line
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Predict
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]  # Probabilities for ROC-AUC

# Metrics
print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_proba))


In [ ]:
from wordcloud import WordCloud

# Example for all text (can also filter by label)
text = ' '.join(df['clean_text'])

wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)

plt.figure(figsize=(15, 7.5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()


In [ ]:
from sklearn.model_selection import cross_val_score

# Define range of C values to try
C_values = [0.01, 0.1, 1, 10]

# Store results
results = []

print("C value | Mean F1 score | Std Dev")
print("-" * 32)

for C_val in C_values:
    model = LogisticRegression(C=C_val, solver='liblinear',class_weight='balanced', max_iter=1000, random_state=42)

    # Use original X_train and y_train instead of balanced ones
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_macro')

    mean_f1 = np.mean(cv_scores).round(4)
    std_f1 = np.std(cv_scores).round(4)

    results.append((C_val, mean_f1, std_f1))

    print(f"{C_val:<6} | {mean_f1:<13} | {std_f1}")

# Optional: find best C
best = max(results, key=lambda x: x[1])
print("\nBest C based on mean F1:", best[0])


In [ ]:
# Unpack automatically (no hardcoding!)
C_vals, mean_f1_scores, std_devs = zip(*results)

# Plot automatically
plt.figure(figsize=(10, 6))
plt.errorbar(C_vals, mean_f1_scores, yerr=std_devs, fmt='-o', capsize=5, color='blue', label='Mean F1 Score')

plt.xscale('log')
plt.title('Effect of Regularization (C) on F1 Score')
plt.xlabel('C Value (log scale)')
plt.ylabel('Mean F1 Score')
plt.xticks(C_vals, C_vals)
plt.ylim(0.9, 1.0)
plt.grid(True, which='both', linestyle='--', linewidth=0.7)
plt.legend()
plt.show()


# 🔹 Model 2: Logistic Regression (TF-IDF, Tuned C=10)

In [ ]:
# Train the final model with C=10
final_model = LogisticRegression(C=10, solver='liblinear', class_weight='balanced', max_iter=1000, random_state=42)
final_model.fit(X_train, y_train)

# Evaluate on the test set
y_pred = final_model.predict(X_test)

# Classification Report on Test Set
from sklearn.metrics import classification_report, confusion_matrix

print("Classification Report on Test Set:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ROC-AUC Score
from sklearn.metrics import roc_auc_score
roc_auc = roc_auc_score(y_test, final_model.predict_proba(X_test)[:, 1])
print(f"ROC-AUC Score on Test Set: {roc_auc:.4f}")


In [ ]:
# Training score
train_score = final_model.score(X_train, y_train)
print(f"Training accuracy: {train_score:.4f}")

# Testing score
test_score = final_model.score(X_test, y_test)
print(f"Testing accuracy: {test_score:.4f}")


In [ ]:
import joblib
joblib.dump(final_model, '/content/drive/My Drive/FYP/final_logistic_regression_model.pkl')

In [ ]:
tfidf.get_feature_names_out()[:20]

In [ ]:
tfidf.get_params()

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)

In [ ]:
import joblib

# Define folder path
folder_path = '/content/drive/My Drive/FYP/'

# ✅ Save cleaned data
df.to_csv(folder_path + 'cleaned_data.csv', index=False)
print("✅ Cleaned data saved as 'cleaned_data.csv'")

# ✅ TF-IDF vectorization
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), max_df=0.95, min_df=2)
X = tfidf.fit_transform(df['clean_text'])

# ✅ Save TF-IDF matrix + vectorizer
joblib.dump(X, folder_path + 'tfidf_matrix.pkl')
joblib.dump(tfidf, folder_path + 'tfidf_vectorizer.pkl')
print("✅ TF-IDF matrix and vectorizer saved")

# ✅ Save labels (optional but useful)
df['label'].to_csv(folder_path + 'labels.csv', index=False)
print("✅ Labels saved as 'labels.csv'")


In [ ]:
# Define folder path
folder_path = '/content/drive/My Drive/FYP/'

# ✅ Save the trained model in FYP folder
joblib.dump(final_model, folder_path + 'logistic_regression_model.pkl')
print("✅ Logistic Regression model saved as 'logistic_regression_model.pkl'")


In [ ]:
# import joblib
# # Load cleaned data
# df = pd.read_csv('cleaned_data.csv')

# # Load TF-IDF matrix (X)
# X = joblib.load('tfidf_matrix.pkl')

# # Load TF-IDF vectorizer (useful if you want to transform new data later)
# tfidf = joblib.load('tfidf_vectorizer.pkl')

# # Load labels (y)
# y = pd.read_csv('labels.csv').values.ravel()  # Convert DataFrame → 1D array

# # Load trained Logistic Regression model
# final_model = joblib.load('logistic_regression_model.pkl')

# print("✅ All saved files loaded successfully!")


# DisTilBERT Implementation

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

In [ ]:
# Load dataset
df = pd.read_csv(folder_path + 'cleaned_text_with_token_reduction.csv')
df = df.dropna(subset=['clean_text'])  # Drop rows with missing clean_text
df = df.reset_index(drop=True)         # Reset index after dropping rows

# Sample smaller dataset
sampled_df = df.sample(n=5000, random_state=42).reset_index(drop=True)



In [ ]:
df

In [ ]:
sampled_df

In [ ]:
df.shape

In [ ]:
sampled_df.shape

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
df.groupby('subject').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

## Visualize the Label

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
df.groupby('label').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
# Split train/test
X_train, X_test, y_train, y_test = train_test_split(df['clean_text'], df['label'], test_size=0.2, stratify=df['label'], random_state=42)


In [ ]:
from transformers import logging
logging.set_verbosity_error()  # suppress warnings


In [ ]:
# Load DistilBERT tokenizer + model
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = AutoModel.from_pretrained('distilbert-base-uncased')


## 🔍 What is Embedding Extraction?

**Embedding extraction** converts text into **numerical vectors** (embeddings) that capture the meaning and context of the text. These vectors help machine learning models process and understand language.

### 🧠 Why It Matters

ML models need numbers, not text. Embeddings provide a semantic-rich numeric form of text, enabling better predictions (e.g., distinguishing Fake vs. True news).

### 📌 Example

> **"Government announces new tax reforms."**  
→ `[0.12, -0.45, 0.87, ..., 0.04]` (e.g., 768-dim vector from BERT)

Similar texts → similar vectors.

### 🛠️ In This Project

We use a **pre-trained BERT** model and extract the `[CLS]` token embedding:

```python
embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()


In [ ]:
import torch
from tqdm import tqdm
import numpy as np
import time

torch.set_float32_matmul_precision('high')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model.to(device)
bert_model.eval()

start_time = time.time()

def get_embeddings(texts, batch_size=128, max_length=128):
    all_embeddings = []
    total_batches = (len(texts) + batch_size - 1) // batch_size

    for i in tqdm(range(0, len(texts), batch_size), desc='Extracting Embeddings', total=total_batches):
        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts.tolist(),
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            outputs = bert_model(**inputs)

        embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(embeddings)

        # Clean up GPU memory
        del inputs, outputs
        torch.cuda.empty_cache()

    return np.vstack(all_embeddings)

X_train_embeddings = get_embeddings(X_train, batch_size=128, max_length=128)
X_test_embeddings = get_embeddings(X_test, batch_size=128, max_length=128)

end_time = time.time()
minutes = int((end_time - start_time) // 60)
seconds = int((end_time - start_time) % 60)
print(f"✅ Total execution time: {minutes} minutes and {seconds} seconds")


In [ ]:
# # Optional: Save for reuse later
# np.save('X_train_embeddings.npy', X_train_embeddings)
# np.save('X_test_embeddings.npy', X_test_embeddings)


# 🔹 Model 3: BERT Embeddings + Logistic Regression

In [ ]:
# Train Logistic Regression
clf = LogisticRegression(class_weight='balanced', max_iter=1000, n_jobs=-1)

clf.fit(X_train_embeddings, y_train)

In [ ]:
# Evaluate
y_pred = clf.predict(X_test_embeddings)
print("Classification Report:\n", classification_report(y_test, y_pred))

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
print("Confusion Matrix:\n", cm)

## Visualize the Embeddings

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained('distilbert-base-uncased')
print(model)


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


In [ ]:
print(f"X_train_embeddings shape: {X_train_embeddings.shape}")
print(f"X_test_embeddings shape: {X_test_embeddings.shape}")


In [ ]:
print(f"Train Accuracy: {clf.score(X_train_embeddings, y_train):.4f}")
print(f"Test Accuracy:  {clf.score(X_test_embeddings, y_test):.4f}")


In [ ]:
import joblib

folder_path = '/content/drive/My Drive/FYP/'

# Save model
clf_filename = folder_path + "fake_news_logreg_model.joblib"
joblib.dump(clf, clf_filename)

# Save tokenizer
tokenizer.save_pretrained(folder_path + 'fake_news_tokenizer')


In [ ]:
import joblib

folder_path = '/content/drive/My Drive/FYP/'

# Save trained Logistic Regression classifier
joblib.dump(clf, folder_path + "trained_logistic_regression.pkl")

# Save tokenizer for reproducibility
tokenizer.save_pretrained(folder_path + "saved_tokenizer")


# Fine-Tune DistilBERT (Full End-to-End)

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import sys
print(sys.version)

In [ ]:
import transformers
print(transformers.__version__)
print([x for x in dir(transformers) if 'DistilBert' in x])

In [ ]:
# ─── Imports ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)

In [ ]:
# ─── Reproducibility ──────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ─── Device ───────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# ─── Load Dataset ─────────────────────────────────────────────
df = pd.read_csv("/content/drive/My Drive/FYP/cleaned_text_with_token_reduction.csv")

df = df.dropna(subset=["clean_text"])
df = df.reset_index(drop=True)

# Encode labels
df["label_id"] = df["label"].map({"TRUE":1, "FAKE":0})

print("Dataset size:", len(df))
print(df["label_id"].value_counts())

In [ ]:
df

In [ ]:
df['clean_text']

In [ ]:
df["label_id"]

In [ ]:
# ─── Train/Validation Split FIRST (important) ─────────────────
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label_id"],
    random_state=SEED
)

In [ ]:
# ─── Remove Cross-Split Duplicates (Leakage Protection) ───────
train_texts_set = set(train_df["clean_text"])

val_df = val_df[~val_df["clean_text"].isin(train_texts_set)]

train_df = train_df.drop_duplicates("clean_text")
val_df   = val_df.drop_duplicates("clean_text")

print("Train size:", len(train_df))
print("Validation size:", len(val_df))

train_texts  = train_df["clean_text"].tolist()
train_labels = train_df["label_id"].tolist()

val_texts  = val_df["clean_text"].tolist()
val_labels = val_df["label_id"].tolist()

In [ ]:
# ─── Compute Class Weights (TRAIN ONLY) ───────────────────────
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

In [ ]:
# ─── Tokenization ─────────────────────────────────────────────
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

MAX_LEN = 256

def tokenize(texts):
    return tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt"
    )

train_encodings = tokenize(train_texts)
val_encodings   = tokenize(val_texts)

In [ ]:
# ─── Dataset Class ────────────────────────────────────────────
class NewsDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

        return item

In [ ]:
# ─── DataLoaders ──────────────────────────────────────────────
batch_size = 32

train_dataset = NewsDataset(train_encodings, train_labels)
val_dataset   = NewsDataset(val_encodings, val_labels)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    pin_memory=True
)

In [ ]:
# ─── Load Model ───────────────────────────────────────────────
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model.to(device)

In [ ]:
# ─── Optimizer & Scheduler ────────────────────────────────────
epochs = 4

total_steps = len(train_loader) * epochs
warmup_steps = int(0.1 * total_steps)

optimizer = AdamW(
    model.parameters(),
    lr=1e-5,
    weight_decay=0.01
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

loss_fn = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
from tqdm import tqdm

best_val_loss = float("inf")
patience = 2
patience_counter = 0
best_model = None

history = {"accuracy":[], "val_accuracy":[], "loss":[], "val_loss":[]}

for epoch in range(epochs):

    print(f"\nEpoch {epoch+1}/{epochs}")

    # ── TRAIN ──
    model.train()

    train_loss_total = 0
    correct = 0
    total = 0

    train_bar = tqdm(train_loader, desc="Training", leave=False)

    for batch in train_bar:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = loss_fn(outputs.logits, labels)

        loss.backward()

        optimizer.step()
        scheduler.step()

        train_loss_total += loss.item()

        preds = torch.argmax(outputs.logits, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

        train_bar.set_postfix(
            loss=loss.item(),
            acc=correct/total
        )

    train_acc = correct / total
    train_loss = train_loss_total / len(train_loader)

    # ── VALIDATION ──
    model.eval()

    val_loss_total = 0
    val_correct = 0
    val_total = 0

    val_bar = tqdm(val_loader, desc="Validation", leave=False)

    with torch.no_grad():

        for batch in val_bar:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = loss_fn(outputs.logits, labels)

            val_loss_total += loss.item()

            preds = torch.argmax(outputs.logits, dim=1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

            val_bar.set_postfix(
                loss=loss.item(),
                acc=val_correct/val_total
            )

    val_acc = val_correct / val_total
    val_loss = val_loss_total / len(val_loader)

    history["accuracy"].append(train_acc)
    history["val_accuracy"].append(val_acc)
    history["loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    print(
        f"Train Loss {train_loss:.4f} | Train Acc {train_acc:.4f} | "
        f"Val Loss {val_loss:.4f} | Val Acc {val_acc:.4f}"
    )

    # Early stopping
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_model = model.state_dict()
        patience_counter = 0

    else:

        patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered")
            break

model.load_state_dict(best_model)

In [ ]:
# ─── Evaluation ───────────────────────────────────────────────
model.eval()

all_preds = []
all_probs = []

with torch.no_grad():

    for batch in val_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probs = torch.softmax(outputs.logits, dim=1)

        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs[:,1].cpu().numpy())

print("\nClassification Report\n")

print(classification_report(
    val_labels,
    all_preds,
    target_names=["FAKE","TRUE"]
))

print("\nConfusion Matrix\n")

print(confusion_matrix(val_labels, all_preds))

In [ ]:
# ─── Save Model ───────────────────────────────────────────────
save_path = "/content/drive/My Drive/FYP/final_distilbert_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved successfully.")

In [ ]:
print(model)

In [ ]:
# ─── Plots ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

epoch_range = range(1, len(history["accuracy"]) + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epoch_range, history["accuracy"],     'b-', label='Training Accuracy')
plt.plot(epoch_range, history["val_accuracy"], 'g-', label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epoch_range, history["loss"],     'b-', label='Training Loss')
plt.plot(epoch_range, history["val_loss"], 'g-', label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs'); plt.ylabel('Loss')
plt.legend(); plt.grid(True)

plt.tight_layout()
plt.show()

# ─── ROC Curve ────────────────────────────────────────────────
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(val_labels, all_probs)
roc_auc     = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='grey', lw=1, linestyle='--')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Fake News Detection')
plt.legend(loc="lower right"); plt.grid(True)
plt.show()

In [ ]:
# ─── Sanity Tests ─────────────────────────────────────────────
print("\n🚀 Running final sanity tests before Viva...")

# 1. Dataset tests
assert df['clean_text'].isna().sum() == 0, "Found NaNs in clean_text!"
assert df['label_id'].isin([0, 1]).all(), "Label IDs not correctly mapped!"
print("✅ Dataset test passed: no NaNs, labels mapped 0/1.")

# 2. Tokenization shape tests
assert train_encodings['input_ids'].shape[1] == MAX_LEN, "Tokenization failed: wrong max length."
assert len(train_encodings['input_ids']) == len(train_texts), "Mismatch in token count."
print("✅ Tokenization test passed: shapes match.")

# 3. Tiny batch overfit test
tiny_texts  = train_texts[:20]
tiny_labels = train_labels[:20]
tiny_enc    = tokenize(tiny_texts)
tiny_ds     = NewsDataset(tiny_enc, tiny_labels)
tiny_loader = DataLoader(tiny_ds, batch_size=4)

model_test  = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
).to(device)
opt_test = AdamW(model_test.parameters(), lr=1e-5)

model_test.train()
for _ in range(3):
    for batch in tiny_loader:
        opt_test.zero_grad()
        out = model_test(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            labels=batch["labels"].to(device)
        )
        out.loss.backward()
        opt_test.step()

print(f"✅ Tiny batch overfit test passed: final loss {out.loss.item():.4f}")

# 4. Accuracy check
assert np.mean(np.array(all_preds) == np.array(val_labels)) > 0.90, "Validation accuracy too low!"
print("✅ Validation accuracy above 90%.")

# 5. Model reload test
save_path = "/content/drive/My Drive/FYP/final_distilbert_model"

model_reloaded     = DistilBertForSequenceClassification.from_pretrained(save_path).to(device)
tokenizer_reloaded = DistilBertTokenizerFast.from_pretrained(save_path)

sample_input  = ["This is a genuine news report.", "Breaking: celebrity endorses fake investment!"]
sample_tokens = tokenizer_reloaded(
    sample_input, truncation=True, padding="max_length",
    max_length=MAX_LEN, return_tensors="pt"
).to(device)

model_reloaded.eval()
with torch.no_grad():
    sample_logits = model_reloaded(**sample_tokens).logits

sample_preds = torch.argmax(sample_logits, dim=1).cpu().numpy()
print(f"✅ Reload test: Predictions on sample: {sample_preds}")

print("\n🎉 All tests passed. Ready for Viva!")

# Fine-Tuned DistilBERT Testing

In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
import torch
import torch.nn.functional as F
import numpy as np

# ─── Device ───────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─── Load trained tokenizer & model ───────────────────────────
tokenizer = DistilBertTokenizerFast.from_pretrained("/content/drive/My Drive/FYP/final_distilbert_model")
model = DistilBertForSequenceClassification.from_pretrained("/content/drive/My Drive/FYP/final_distilbert_model")
model.to(device)
model.eval()

# ─── Helper function ──────────────────────────────────────────
label_map = {0: "FAKE", 1: "TRUE"}

def predict(samples):
    inputs = tokenizer(
        samples,
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits

    probs       = F.softmax(logits, dim=1).cpu().numpy()
    pred_classes = np.argmax(probs, axis=1)

    return pred_classes, probs

## Political/Government News Samples

In [ ]:
test_samples = [
    """The government unveiled a comprehensive new education initiative today, allocating over
    50 billion rupees towards improving infrastructure in rural schools. Officials stated that
    this plan aims to reduce dropout rates and increase literacy levels over the next decade.""",

    """According to widely shared social media posts, the President recently confessed during
    a live television broadcast that the last election was manipulated by foreign operatives.
    However, independent fact-checkers and multiple international observers have found no
    evidence to support these claims.""",

    """Left-wing political coalitions are pushing aggressively for new climate legislation,
    seeking to impose stricter carbon emission limits across all industries. Critics argue
    this will harm the manufacturing sector, while supporters believe it is crucial to
    mitigate the effects of climate change.""",

    """A series of articles circulating online allege that the government passed a secret
    bill authorizing mass electronic surveillance of citizens without any judicial oversight.
    Legal experts have called these reports baseless, pointing out that no such legislation
    exists on parliamentary records."""
]

pred_classes, probs = predict(test_samples)

for text, pred, prob in zip(test_samples, pred_classes, probs):
    label      = label_map[pred]
    confidence = prob[pred]
    print(f"\n📰 News: \"{text[:80]}...\"")
    print(f"✅ Prediction: {label} (confidence: {confidence:.4f})")

## Rigorous Test Samples

In [ ]:
test_samples = [
    "The sun rises in the east and sets in the west.",
    "Pakistan's capital city is Islamabad.",
    "Scientists have confirmed that drinking bleach cures all diseases instantly.",
    "The government has built secret mind-control towers in every city that beam signals into your brain.",
    "A new study claims that eating only ice cream triples your lifespan.",
    "Reports say aliens are currently negotiating with world leaders to take over Earth.",
    "The Prime Minister attended the climate summit held in Dubai last week.",
    "The new metro train project aims to reduce traffic congestion in Lahore."
]

pred_classes, probs = predict(test_samples)

for text, pred, prob in zip(test_samples, pred_classes, probs):
    print(f"\n📰 \"{text[:80]}...\"")
    print(f"✅ Prediction: {label_map[pred]} (conf: {prob[pred]:.4f}) | FAKE: {prob[0]:.4f}, TRUE: {prob[1]:.4f}")

## Extreme Edge Cases

In [ ]:
edge_samples = [
    "Aliens elected as next Prime Minister.",
    "Vaccines turn humans into lizards overnight.",
    "Earth is flat, NASA confirms after decades of lies.",
    "Chocolate cake can repair broken bones instantly.",
    "World to end tomorrow at 3 PM.",
    "Government bans all breathing, citizens must pay for air.",
    "Cats discovered to speak fluent French.",
    "Breaking: scientists find oceans are made of lemonade.",
    "Prime Minister launches new economic plan to boost exports.",
    "Local farmers report record wheat yields this season."
]

pred_classes, probs = predict(edge_samples)

for text, pred, prob in zip(edge_samples, pred_classes, probs):
    print(f"\n📰 \"{text}\"")
    print(f"✅ Prediction: {label_map[pred]} (conf: {prob[pred]:.4f}) | FAKE: {prob[0]:.4f}, TRUE: {prob[1]:.4f}")

## Political Paragraphs (Edge Cases)

In [ ]:
edge_samples = [
    """The government unveiled a comprehensive new education initiative today, allocating over
    50 billion rupees towards improving infrastructure in rural schools. Officials stated that
    this plan aims to reduce dropout rates and increase literacy levels over the next decade.""",

    """According to widely shared social media posts, the President recently confessed during
    a live television broadcast that the last election was manipulated by foreign operatives.
    However, independent fact-checkers and multiple international observers have found no
    evidence to support these claims.""",

    """Left-wing political coalitions are pushing aggressively for new climate legislation,
    seeking to impose stricter carbon emission limits across all industries. Critics argue
    this will harm the manufacturing sector, while supporters believe it is crucial to
    mitigate the effects of climate change.""",

    """A series of articles circulating online allege that the government passed a secret
    bill authorizing mass electronic surveillance of citizens without any judicial oversight.
    Legal experts have called these reports baseless, pointing out that no such legislation
    exists on parliamentary records."""
]

pred_classes, probs = predict(edge_samples)

for text, pred, prob in zip(edge_samples, pred_classes, probs):
    print(f"\n📰 \"{text[:80]}...\"")
    print(f"✅ Prediction: {label_map[pred]} (conf: {prob[pred]:.4f}) | FAKE: {prob[0]:.4f}, TRUE: {prob[1]:.4f}")

# Final Comparative Analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ==============================
# Confusion Matrices (REAL RESULTS)
# ==============================

cm_lr = np.array([[2459, 46],
                  [42, 4193]])

cm_lr_tuned = np.array([[2476, 29],
                        [26, 4209]])

cm_bert_embed = np.array([[2478, 22],
                          [25, 4210]])

cm_distilbert = np.array([[2498, 2],
                          [2, 4230]])

models = {
    "LogReg TF-IDF": cm_lr,
    "LogReg TF-IDF (C=10)": cm_lr_tuned,
    "LogReg BERT Embeddings": cm_bert_embed,
    "Fine-Tuned DistilBERT": cm_distilbert
}

# ==============================
# Function to compute metrics
# ==============================

def compute_metrics(cm):
    
    TN, FP, FN, TP = cm.ravel()
    
    y_true = ([0]*TN + [0]*FP + [1]*FN + [1]*TP)
    y_pred = ([0]*TN + [1]*FP + [0]*FN + [1]*TP)
    
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    return accuracy, precision, recall, f1


# ==============================
# Build comparison table
# ==============================

results = []

for name, cm in models.items():
    
    acc, prec, rec, f1 = compute_metrics(cm)
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("Accuracy", ascending=False)

print("\nFinal Model Comparison\n")
print(results_df.round(4))


# ==============================
# Plot Confusion Matrices
# ==============================

plt.figure(figsize=(12,10))

for i,(name,cm) in enumerate(models.items(),1):
    
    plt.subplot(2,2,i)
    sns.heatmap(cm,
                annot=True,
                fmt="d",
                cmap="Blues",
                cbar=False)
    
    plt.title(name)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

plt.tight_layout()
plt.show()


# ==============================
# Bar Chart Comparison
# ==============================

results_df.set_index("Model")[["Accuracy","F1 Score","Precision","Recall"]].plot(
    kind="bar",
    figsize=(10,6)
)

plt.title("Fake News Detection Model Comparison")
plt.ylabel("Score")
plt.xticks(rotation=20)
plt.ylim(0.97,1.01)
plt.grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

# Performane Comparison

In [ ]:
import pandas as pd

data = {
    "Model": [
        "LogReg (TF-IDF)",
        "LogReg (TF-IDF, C=10)",
        "LogReg (DistilBERT Embeddings)",
        "Fine-Tuned DistilBERT"
    ],
    "Accuracy": [0.99, 0.99, 0.99, 1.00],
    "F1-Score": [0.99, 0.99, 0.99, 1.00],
    "ROC-AUC": [0.9989, 0.9991, None, 1.00]
}

df = pd.DataFrame(data)
print(df)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Remove None values for ROC-AUC
df_plot = df.copy()
df_plot["ROC-AUC"] = df_plot["ROC-AUC"].fillna(0)

bar_width = 0.22
r1 = np.arange(len(df_plot))
r2 = [x + bar_width for x in r1]
r3 = [x + bar_width for x in r2]

plt.figure(figsize=(12,6))
plt.bar(r1, df_plot["Accuracy"], color='skyblue', width=bar_width, label='Accuracy')
plt.bar(r2, df_plot["F1-Score"], color='lightgreen', width=bar_width, label='F1-Score')
plt.bar(r3, df_plot["ROC-AUC"], color='salmon', width=bar_width, label='ROC-AUC (0 if NA)')

plt.xlabel('Model')
plt.ylabel('Score')
plt.title('Comparison of Models for Fake News Detection')
plt.xticks([r + bar_width for r in range(len(df_plot))], df_plot["Model"], rotation=15)
plt.ylim(0.95, 1.01)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
from tabulate import tabulate

table_data = [
    ["LogReg (TF-IDF)", "99.0%", "~0.99", "0.9989"],
    ["LogReg (TF-IDF, C=10)", "99.0%", "~0.99", "0.9991"],
    ["LogReg (DistilBERT Embeddings)", "99.0%", "~0.99", "-"],
    ["Fine-Tuned DistilBERT", "100.0%", "~1.00", "1.00"]
]

headers = ["Model", "Accuracy", "F1-Score", "ROC-AUC"]

print(tabulate(table_data, headers=headers, tablefmt="fancy_grid"))
